In [77]:
import json, pickle
import numpy as np
import pandas as p
import math as m

%pylab inline

Populating the interactive namespace from numpy and matplotlib


`%matplotlib` prevents importing * from pylab and numpy


In [48]:
ds = Dataset.objects.get(pk=19)

In [49]:
with open(ds.static_path('nodes_inv.pickle')) as fp:
    nodes_inv = pickle.load(fp)

In [50]:
nodes_inv_inv = {}
for nid, sids in nodes_inv.iteritems():
    for sid in sids:
        nodes_inv_inv[sid] = nid

In [51]:
layout = json.load(open(ds.static_path('layout.json')))['nodes']
corr = json.load(open(ds.static_path('correlations.json')))['edges']
nodes = {n['id']: n for n in json.load(open(ds.static_path('nodes.json')))['nodes']}

In [52]:
layout = {l['id']: l for l in layout}
data = {}
for c in corr:
    data.setdefault(c['s'], set()).add((c['t'], c['w']))
    data.setdefault(c['t'], set()).add((c['s'], c['w']))

In [79]:
def coor(x):
    x = layout[x]
    return (x['x'], x['y'])

def dist(a, b):
    return np.sqrt(((a[0] - b[0])**2) + ((a[1] - b[1])**2))

def angle_trunc(a):
    while a < 0.0:
        a += m.pi * 2
    return m.degrees(a)

In [115]:
for node, connections in data.iteritems():
#     if node not in (216, 1): continue
    if node not in layout: continue
    
    a = coor(node)
    angles = []
    dists = []
    for con, w in connections:
        if con not in layout: continue
        
        b = coor(con)
        d = dist(a,b)
        ang = angle_trunc(m.atan2((a[0] - b[0]), (a[1] - b[1])))
        
        if d < 200: continue
        
        angles.append(ang)
        dists.append(d)
    
    if len(angles) < 4: continue
    
    angles = p.Series(sorted(angles))
    window = angles.append(angles + 360).rolling(8).std().dropna()
    
    print nodes[node]['label'], window.median() #, list(window)
#     break

ria1-ts 35.1590290055
YPT7 175.446426623
rnt1-ts 153.850409057
LEO1 99.3857748927
MRPL20 12.8703339929
YDR239C 217.104587997
tfc4-5001 151.653870974
cdc28-4 120.468495754
ERP3 228.027020211
taf9-ts2 153.271412379
ATP19 95.4676010486
FAR7 173.207976513
DAN1 177.522669604
ELP3 67.4454887386
URA2 143.110284464
PMT2 195.304028594
YOR072W-B 96.7164989574
srp72-5003 137.475880681
EAF7 191.21212025
ATG12 227.149220146
HSL1 110.164403733
taf13-2 142.308175147
RAD24 19.7914914202
rcl1-5004 182.768536319
GID8 226.583983266
sec15-1 80.4312494102
RPS4B 194.659359212
CCW14 119.600492673
afg2-18 151.143395999
rpt3-1 164.369359866
dad2-9 89.2111872191
ATP5 150.020605804
DIA2 77.9496083209
YDR306C 178.60787363
BUB1 5.10484093793
ntf2-5001 145.111308312
cdc2-7 157.746443952
cct6-18-supp1 180.687207818
EGH1 198.494614965
KEL3 136.715972345
BRE2 125.576123093
pkc1-2 180.484983577
ALO1 65.9519071113
isc1-supp1 217.211145697
erg27-5001 168.313811388
IMP2' 225.193394623
PET10 137.753250548
ADI1 182.65140676

In [117]:
xs = []
ys = []
for n in json.load(open(ds.static_path('layout.json')))['nodes']:
    xs.append(n['x'])
    ys.append(n['y'])

In [120]:
min(xs) - max(xs)

-3360.6221923828202

In [121]:
min(ys) - max(ys)

-3344.29296875